# 허깅페이스 도란스뽀마 설치 및 환경 구성

In [1]:
!pip install transformers
!pip install accelerate

In [2]:
!python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('I love you'))"

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████████████████| 104/104 [00:00<00:00, 3706.88it/s]
[{'label': 'POSITIVE', 'score': 0.9998656511306763}]


### 허깅 페이스 데이터세트 불러오기

In [3]:
!pip install datasets

In [4]:
import datasets
from datasets import load_dataset

huggingface_mrpc_dataset = load_dataset('glue', 'mrpc')
print(huggingface_mrpc_dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


In [5]:
train = huggingface_mrpc_dataset['train']
cols = train.column_names
cols

['sentence1', 'sentence2', 'label', 'idx']

In [6]:
for i in range(5):
    for col in cols:
        print(col, ":", train[col][i])
    print('\n')

sentence1 : Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
sentence2 : Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
label : 1
idx : 0


sentence1 : Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
sentence2 : Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .
label : 0
idx : 1


sentence1 : They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
sentence2 : On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .
label : 1
idx : 2


sentence1 : Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
sentence2 : Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
label : 0

### 커스텀 데이터세트 만들기

In [7]:
import pandas as pd
from datasets import Dataset

def parse_mrpc_file(file_path):
    """MRPC 파일을 안전하게 파싱하는 함수"""
    data = {
        'Quality': [],
        '#1 ID': [],
        '#2 ID': [], 
        '#1 String': [],
        '#2 String': []
    }
    
    with open(file_path, 'r', encoding='utf-8') as f:
        # 헤더 스킵
        next(f)
        
        for line_num, line in enumerate(f, 1):
            try:
                parts = line.strip().split('\t')
                if len(parts) >= 5:
                    # 5개 컬럼으로 분할 (마지막 탭들은 모두 마지막 컬럼에 포함)
                    quality = int(parts[0])
                    id1 = int(parts[1]) 
                    id2 = int(parts[2])
                    string1 = parts[3]
                    string2 = '\t'.join(parts[4:])  # 나머지 모든 부분을 합침
                    
                    data['Quality'].append(quality)
                    data['#1 ID'].append(id1)
                    data['#2 ID'].append(id2)
                    data['#1 String'].append(string1)
                    data['#2 String'].append(string2)
                else:
                    print(f"Line {line_num}: Invalid format, skipping")
            except Exception as e:
                print(f"Line {line_num}: Error {e}, skipping")
    
    return pd.DataFrame(data)

# 로컬 파일에서 MRPC 데이터 읽기
train_df = parse_mrpc_file('data/msr_paraphrase_train.txt')
test_df = parse_mrpc_file('data/msr_paraphrase_test.txt')

In [8]:
# 데이터 구조 확인
print("Train dataset columns:", train_df.columns.tolist())
print("Train dataset shape:", train_df.shape)
print("\nFirst 5 examples:")

for i in range(5):
    row = train_df.iloc[i]
    for col in train_df.columns:
        print(f"{col}: {row[col]}")
    print('\n')

Train dataset columns: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String']
Train dataset shape: (4076, 5)

First 5 examples:
Quality: 1
#1 ID: 702876
#2 ID: 702977
#1 String: Amrozi accused his brother, whom he called "the witness", of deliberately distorting his evidence.
#2 String: Referring to him as only "the witness", Amrozi accused his brother of deliberately distorting his evidence.


Quality: 0
#1 ID: 2108705
#2 ID: 2108831
#1 String: Yucaipa owned Dominick's before selling the chain to Safeway in 1998 for $2.5 billion.
#2 String: Yucaipa bought Dominick's in 1995 for $693 million and sold it to Safeway for $1.8 billion in 1998.


Quality: 1
#1 ID: 1330381
#2 ID: 1330521
#1 String: They had published an advertisement on the Internet on June 10, offering the cargo for sale, he added.
#2 String: On June 10, the ship's owners had published an advertisement on the Internet, offering the explosives for sale.


Quality: 0
#1 ID: 3344667
#2 ID: 3344648
#1 String: Around 0335 GMT, 

In [9]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# train 데이터를 train(80%)과 validation(20%)으로 분할
train_indices = list(range(len(train_dataset['Quality'])))
train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42, 
                                      stratify=train_dataset['Quality'])

# validation 데이터셋 생성
validation_dataset = {}
for key in train_dataset.keys():
    validation_dataset[key] = [train_dataset[key][i] for i in val_idx]

# train 데이터셋 업데이트 (validation으로 사용된 데이터 제거)
train_dataset_final = {}
for key in train_dataset.keys():
    train_dataset_final[key] = [train_dataset[key][i] for i in train_idx]

# 허깅페이스 Dataset 객체로 변환
train_hf_dataset = Dataset.from_dict(train_dataset_final)
validation_hf_dataset = Dataset.from_dict(validation_dataset)
test_hf_dataset = Dataset.from_dict(test_dataset)

# DatasetDict 생성 (이게 허깅페이스의 표준 방식)
customized_mrpc_dataset = DatasetDict({
    'train': train_hf_dataset,
    'validation': validation_hf_dataset,
    'test': test_hf_dataset
})

# 결과 출력
print("DatasetDict({")
for split_name, split_data in customized_mrpc_dataset.items():
    print(f"    {split_name}: Dataset({{")
    print(f"        features: {list(split_data.features.keys())},")
    print(f"        num_rows: {split_data.num_rows}")
    print("    })")
print("})")

print("\n데이터셋 정보 확인!")
print(f"Train dataset shape: ({customized_mrpc_dataset['train'].num_rows}, {len(customized_mrpc_dataset['train'].features)})")
print(f"Validation dataset shape: ({customized_mrpc_dataset['validation'].num_rows}, {len(customized_mrpc_dataset['validation'].features)})")
print(f"Test dataset shape: ({customized_mrpc_dataset['test'].num_rows}, {len(customized_mrpc_dataset['test'].features)})")

print("Dataset 생성 완료!")
print(f"Train samples: {len(customized_mrpc_dataset['train'])}")
print(f"Validation samples: {len(customized_mrpc_dataset['validation'])}")
print(f"Test samples: {len(customized_mrpc_dataset['test'])}")

# 첫 번째 샘플 확인
print(f"\n첫 번째 train 샘플:")
print(customized_mrpc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 3260
    })
    validation: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 816
    })
    test: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 1725
    })
})

데이터셋 정보 확인!
Train dataset shape: (3260, 5)
Validation dataset shape: (816, 5)
Test dataset shape: (1725, 5)
Dataset 생성 완료!
Train samples: 3260
Validation samples: 816
Test samples: 1725

첫 번째 train 샘플:
{'Quality': 0, '#1 ID': 2121506, '#2 ID': 2121285, '#1 String': 'The festival kicked off yesterday one day after the Competition Commission delivered its final verdict to the Government on the proposed £4.1 billion merger.', '#2 String': 'The Competition Commission delivered its verdict yesterday on the proposed merger of the two big ITV players, Carlton and Granada.'}


### 토크나이저와 모델

In [10]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

huggingface_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
huggingface_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def transform(data):
    return huggingface_tokenizer(
        data['sentence1'],
        data['sentence2'],
        truncation = True,
        padding = 'max_length',
        return_token_type_ids = False,
        )

In [12]:
hf_dataset = huggingface_mrpc_dataset.map(transform, batched=True)

# train & validation & test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

In [13]:
# cell 11에서 이미 customized_mrpc_dataset 만들어놨으니 그거 그대로 갖다씀
# val_dataset = test_dataset.copy() 같은 짓 하면 안됨 - 데이터 누수라고 함.

# label 컬럼 이름 맞춰주기 (HF trainer가 자동으로 라벨 인식하려면 'label' 또는 'labels' 여야됨)
# 'Quality'를 'label'로 리네임
customized_mrpc_dataset = customized_mrpc_dataset.rename_column('Quality', 'label')

# 커스텀 transform - 컬럼명만 다름 (#1 String, #2 String)
def transform_custom(batch):
    return huggingface_tokenizer(
        batch['#1 String'],
        batch['#2 String'],
        truncation=True,
        padding='max_length',
        return_token_type_ids=False,
    )

# 토큰화 일괄 적용 (DatasetDict는 map 한방으로 다 처리됨)
customized_mrpc_dataset = customized_mrpc_dataset.map(transform_custom, batched=True)

# split 꺼내쓰기 - 변수명도 명확하게
custom_train_dataset = customized_mrpc_dataset['train']
custom_val_dataset = customized_mrpc_dataset['validation']
custom_test_dataset = customized_mrpc_dataset['test']

print('custom dataset 토큰화 완료')
print(f'train: {len(custom_train_dataset)}, val: {len(custom_val_dataset)}, test: {len(custom_test_dataset)}')
print(f'features: {custom_train_dataset.column_names}')


Map:   0%|          | 0/3260 [00:00<?, ? examples/s]

Map:   0%|          | 0/816 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

custom dataset 토큰화 완료
train: 3260, val: 816, test: 1725
features: ['label', '#1 ID', '#2 ID', '#1 String', '#2 String', 'input_ids', 'attention_mask']


### 트레인 엡발루에이션과 테스트

#### 최적화 개조(진영코드)

In [14]:
import os
import gc
import torch
import numpy as np
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

output_dir = 'transformers'

# fp16은 cuda 있을때만 켜기 - 안그러면 cpu에서 폭발함
use_fp16 = torch.cuda.is_available()
print(f'fp16 사용여부: {use_fp16}')

# 최적화된 TrainingArguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    eval_strategy='epoch',                    # epoch 끝날때마다 평가
    save_strategy='epoch',                    # epoch 끝날때마다 저장
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,                         # 처음 10% 스텝은 warmup - 트랜스포머는 이거 있어야 안정적
    lr_scheduler_type='linear',
    load_best_model_at_end=True,              # 학습 끝나면 best 체크포인트로 자동복원
    metric_for_best_model='f1',               # MRPC 공식 평가지표가 f1 이라 그거 기준으로
    greater_is_better=True,
    save_total_limit=2,                       # 체크포인트 2개만 유지 - 디스크 절약
    fp16=use_fp16,                            # mixed precision - gpu에서 1.5~2배 빠름
    dataloader_num_workers=2,
    dataloader_pin_memory=use_fp16,           # gpu 있을때만 의미있음
    logging_steps=50,
    report_to='none',                         # wandb 같은거 끔
    seed=42,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


fp16 사용여부: True


In [15]:
!pip install evaluate

In [16]:
from evaluate import load
metric = load('glue', 'mrpc')

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references = labels)

In [17]:
trainer = Trainer(
    model=huggingface_model,
    args=training_arguments,
    train_dataset=hf_train_dataset,
    eval_dataset=hf_val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # 2번 연속 안좋아지면 멈춤
)
trainer.train()
print('완료')


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.532212,0.366068,0.838235,0.885017
2,0.425606,0.394669,0.840686,0.888124
3,0.307760,0.551544,0.855392,0.898100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

완료


In [18]:
trainer.evaluate(hf_test_dataset)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.307760,0.583918,3,0.833623,0.877820


{'eval_loss': 0.5839183330535889,
 'eval_accuracy': 0.8336231884057971,
 'eval_f1': 0.877820349084717}

In [19]:
# 메모리 빡세게 비워줍니다 - 그냥 del만 하면 cuda 캐시는 안풀림
del huggingface_model
del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('메모리 정리 완료')


메모리 정리 완료


In [20]:
# Q. 커스텀 데이터셋으로 학습시켜봅시다.
# tf_train_dataset 이런 변수는 없음 - 우리가 만든건 custom_train_dataset 임
# 그리고 output_dir 도 분리해서 hf 학습이랑 안섞이게

huggingface_model_custom = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

# 커스텀 데이터용 별도 output_dir
training_arguments_custom = TrainingArguments(
    output_dir='transformers_custom',          # 분리
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=use_fp16,
    dataloader_num_workers=2,
    dataloader_pin_memory=use_fp16,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer_custom = Trainer(
    model=huggingface_model_custom,
    args=training_arguments_custom,
    train_dataset=custom_train_dataset,        # <-- 여기 tf_train_dataset 아님
    eval_dataset=custom_val_dataset,           # <-- 여기도
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_custom.train()

# 테스트셋으로 최종 평가
print('\n=== 커스텀 테스트셋 최종평가 ===')
print(trainer_custom.evaluate(custom_test_dataset))


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.416882,0.449064,0.805147,0.856109
2,0.310264,0.593444,0.808824,0.865749
3,0.307446,0.639968,0.814951,0.866253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== 커스텀 테스트셋 최종평가 ===


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.307446,0.601245,3,0.825507,0.872620


{'eval_loss': 0.6012453436851501, 'eval_accuracy': 0.8255072463768116, 'eval_f1': 0.8726195514176894}


In [21]:
# Validation 데이터셋 평가 (학습 중 모니터링한 그 셋 - best checkpoint 기준)
print('\n=== 커스텀 validation 셋 평가 ===')
eval_results = trainer_custom.evaluate()
print(eval_results)

# 테스트셋 최종 평가 (학습 중 한번도 안 본 데이터)
print('\n=== 커스텀 test 셋 최종 평가 ===')
print(trainer_custom.evaluate(custom_test_dataset))


=== 커스텀 validation 셋 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.307446,0.639968,3,0.814951,0.866253


{'eval_loss': 0.6399680972099304, 'eval_accuracy': 0.8149509803921569, 'eval_f1': 0.8662533215234721}

=== 커스텀 test 셋 최종 평가 ===


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.307446,0.601245,3,0.825507,0.872620


{'eval_loss': 0.6012453436851501, 'eval_accuracy': 0.8255072463768116, 'eval_f1': 0.8726195514176894}
